# Assignment 2 – RAG
**Students:** Ilay Nisim & Boris Kogan  
**Due date:** 28.4.26

---

## Setup
Install all packages needed for the entire assignment (Tasks 1–7 + Bonus).

In [1]:
import sys
!{sys.executable} -m pip install -q openai pandas openpyxl datasets langchain langchain-community faiss-cpu sentence-transformers pypdf cryptography ragas python-dotenv


In [ ]:
import os
import json
import pandas as pd
from openai import OpenAI
from datasets import load_dataset
from dotenv import load_dotenv

load_dotenv()  # reads .env from the current directory

NEBIUS_API_KEY   = os.environ["NEBIUS_API_KEY"]
NEBIUS_BASE_URL  = "https://api.studio.nebius.ai/v1/"

LLM_MODEL        = "meta-llama/Llama-3.3-70B-Instruct"   # generation (Tasks 1, 4, 5)
JUDGE_MODEL      = "deepseek-ai/DeepSeek-V3.2"         # judge (Tasks 6, 7)
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"               # embeddings (Tasks 3–7)

client = OpenAI(api_key=NEBIUS_API_KEY, base_url=NEBIUS_BASE_URL)

def llm_generate(question: str, system_prompt: str = None, model: str = LLM_MODEL) -> str:
    """Single reusable helper for all LLM calls in this assignment."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": question})
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.1,
        max_tokens=500,
    )
    return response.choices[0].message.content.strip()

print("Setup complete. Model:", LLM_MODEL)

/Users/YOUR_USER/dev/cloud/nebius/lesson5_rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete. Model: meta-llama/Llama-3.3-70B-Instruct


---
## Dataset preparation

Load [FinanceBench](https://huggingface.co/datasets/PatronusAI/financebench), drop `metrics-generated` questions, fix dead doc links, and parse the `evidence` column into flat fields reused across Tasks 3–7.

In [3]:
# Load
df = load_dataset("PatronusAI/financebench", split="train").to_pandas()

# Drop metrics-generated
df = df[df["question_type"] != "metrics-generated"].copy()

# Rename answer → ground_truth
df = df.rename(columns={"answer": "ground_truth"})

# Fix dead doc_link URLs
df["doc_link"] = df["doc_name"].apply(
    lambda name: f"https://raw.githubusercontent.com/patronus-ai/financebench/main/pdfs/{name}.pdf"
)

# Parse evidence column → evidence_page_nums (list[int]) + evidence_texts (list[str])
# The column can be: a JSON string, a list of dicts, or a numpy array of dicts
import numpy as np

def parse_evidence(ev):
    if isinstance(ev, str):
        ev = json.loads(ev)
    if isinstance(ev, np.ndarray):
        ev = ev.tolist()
    if not isinstance(ev, list):
        ev = [ev]
    page_nums, texts = [], []
    for e in ev:
        if isinstance(e, dict):
            page_nums.append(int(e.get("evidence_page_num", -1)))
            texts.append(str(e.get("evidence_text", "")))
        else:
            # numpy structured array row or similar — convert to dict first
            e = dict(e)
            page_nums.append(int(e.get("evidence_page_num", -1)))
            texts.append(str(e.get("evidence_text", "")))
    return page_nums, texts

parsed = df["evidence"].apply(parse_evidence)
df["evidence_page_nums"] = parsed.apply(lambda x: x[0])
df["evidence_texts"]     = parsed.apply(lambda x: x[1])

# Doc metadata lookup: doc_name → {company, doc_period, doc_link}  (used by Task 3)
doc_meta = (
    df[["doc_name", "company", "doc_period", "doc_link"]]
    .drop_duplicates(subset="doc_name")
    .set_index("doc_name")
    .to_dict(orient="index")
)

# Sort
df = df.sort_values("financebench_id").reset_index(drop=True)

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"Rows: {len(df)}  |  Unique docs: {len(doc_meta)}")
print(df["question_type"].value_counts().to_string())
print()
print(df[["financebench_id", "question_type", "question", "doc_name"]].head(3).to_string(index=False))

Rows: 100  |  Unique docs: 42
question_type
domain-relevant    50
novel-generated    50

      financebench_id   question_type                                                                                                                                                                                      question                    doc_name
financebench_id_00005 domain-relevant              Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.            CORNING_2022_10K
financebench_id_00070 domain-relevant Does American Water Works have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why. AMERICANWATERWORKS_2022_10K
financebench_id_00080 domain-relevant               Does Paypal have positive working capital based on FY2022 data? If working capital is not a

---
## Task 1 – naive generation (10 points)

We send the first 5 questions of each type straight into `Llama-3.3-70B-Instruct` — no retrieval, no grounding — and compare the outputs to the ground truth.

In [4]:
# First 5 per type — kept as separate named slices for reuse in Tasks 5, 6, 7
task1_domain = df[df["question_type"] == "domain-relevant"].head(5).copy()
task1_novel  = df[df["question_type"] == "novel-generated"].head(5).copy()
task1_df     = pd.concat([task1_domain, task1_novel]).sort_values("financebench_id").copy()

print(f"{len(task1_df)} questions selected  ({len(task1_domain)} domain-relevant, {len(task1_novel)} novel-generated):")
for _, row in task1_df.head(3).iterrows():
    print(f"  [{row['financebench_id']}] ({row['question_type']}) {row['question'][:120]}")
print("  ...")

10 questions selected  (5 domain-relevant, 5 novel-generated):
  [financebench_id_00005] (domain-relevant) Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric f
  [financebench_id_00070] (domain-relevant) Does American Water Works have positive working capital based on FY2022 data? If working capital is not a useful or rele
  [financebench_id_00080] (domain-relevant) Does Paypal have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric fo
  ...


In [5]:
NAIVE_SYSTEM_PROMPT = (
    "You are a helpful financial analyst. "
    "Answer the question to the best of your knowledge. "
    "Be concise."
)

naive_answers = []
for i, (_, row) in enumerate(task1_df.iterrows()):
    if i < 3:
        print(f"[{row['financebench_id']}] querying...")
    answer = llm_generate(row["question"], system_prompt=NAIVE_SYSTEM_PROMPT)
    naive_answers.append(answer)
    if i < 3:
        print(f"  → {answer[:150]}\n")

task1_df["naive_answer"] = naive_answers
print("Done.")

[financebench_id_00005] querying...
  → Based on Corning's FY2022 data, the company's current assets were $7.85 billion and current liabilities were $4.93 billion. This results in a positive

[financebench_id_00070] querying...
  → Based on American Water Works' FY2022 data, the company's current assets were approximately $2.43 billion, and its current liabilities were around $2.

[financebench_id_00080] querying...
  → Based on PayPal's FY2022 data, the company's working capital is negative. As of December 31, 2022, PayPal's current assets were $63.8 billion, and its

Done.


In [6]:
# Sample results for manual review
pd.set_option("display.max_colwidth", 250)
task1_df[["financebench_id", "question_type", "question", "naive_answer", "ground_truth"]].head(2)

,financebench_id,question_type,question,naive_answer,ground_truth
0,financebench_id_00005,domain-relevant,"Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.","Based on Corning's FY2022 data, the company's current assets were $7.85 billion and current liabilities were $4.93 billion. This results in a positive working capital of $2.92 billion.\n\nWorking capital is a relevant metric for Corning, as it is...",Yes. Corning had a positive working capital amount of $831 million by FY 2022 close. This answer considers only operating current assets and current liabilities that were clearly shown in the balance sheet.
1,financebench_id_00070,domain-relevant,"Does American Water Works have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.","Based on American Water Works' FY2022 data, the company's current assets were approximately $2.43 billion, and its current liabilities were around $2.55 billion. This results in negative working capital of approximately -$120 million.\n\nHowever,...","No, American Water Works had negative working capital of -$1561M in FY 2022."


In [7]:
# Heuristic pre-fill, then manual corrections from our spot-check in the
# discussion below. (The heuristic misfires on 00302 — the naive answer starts
# with "I don't have the exact figures" but then gives a substantive wrong
# answer, so it needs to be overridden to partially correct.)
REFUSE_PHRASES = [
    "i don't have", "i cannot", "i don't know", "i'm unable",
    "i need more", "please provide", "could you clarify",
    "without access", "not available to me",
]

task1_df["verdict"] = task1_df["naive_answer"].apply(
    lambda a: "refused" if any(p in a.lower() for p in REFUSE_PHRASES) else "wrong"
)

# ── Manual corrections (from the spot-check in the Task 1 discussion below) ──
manual_verdicts = {
    "financebench_id_00005": "partially correct",   # Corning WC: right sign, wrong figure
    "financebench_id_00070": "partially correct",   # AWW WC: right sign, order-of-magnitude off
    "financebench_id_00080": "wrong",               # PayPal WC: wrong sign + fabricated figures
    "financebench_id_00206": "correct",             # JPM gross margin N/A — matches GT
    "financebench_id_00215": "partially correct",   # Boeing: direction right, number off
    "financebench_id_00283": "wrong",               # Pfizer-Upjohn: $12B vs $77.78M, ~150x off
    "financebench_id_00288": "refused",             # BestBuy Q2 FY24: past training cutoff
    "financebench_id_00299": "partially correct",   # JPM segment: right entity, wrong sign
    "financebench_id_00302": "partially correct",   # Pfizer PPNE: direction right (heuristic misfires — overriding)
    "financebench_id_00382": "correct",             # MGM Las Vegas EBITDAR
}
for _id, _v in manual_verdicts.items():
    task1_df.loc[task1_df.financebench_id == _id, "verdict"] = _v

print(task1_df["verdict"].value_counts().to_string())

In [8]:
OUT_FILE = "assignment2_naive_generation_ilay_nisim_boris_kogan.xlsx"
task1_df[
    ["financebench_id", "question_type", "question", "naive_answer", "ground_truth", "verdict"]
].to_excel(OUT_FILE, index=False)
print(f"Saved → {OUT_FILE}")

Saved → assignment2_naive_generation_ilay_nisim_boris_kogan.xlsx


### Task 1 – Discussion

**1. Cases where the model refused or asked for more information - why?**  
Only 1 refusal: `financebench_id_00288` (cash drop FY2023 → Q2 FY2024). The model said it lacks access to real-time data — Looks like the reason for the refusal is that Q2 FY2024 falls beyond the model's training cutoff. 

**2. Cases where the model answered confidently - spot-check against the ground truth. Is the answer correct? Partially correct? Totally wrong (hallucinated)?**  
Cases where the model answered confidently divide into **2 correct**, **5 partially correct**, **2 hallucinated**:
- **Correct:** `00206` (JPM gross margin not relevant for a bank — probably because it requires pure financial reasoning, domain-relevant) and `00382` (MGM Las Vegas highest EBITDAR — well-known fact).
- **Partially correct:** right direction or entity but wrong numbers. E.g. `00005` Corning WC positive ✓ but $3.02B vs $831M (model used all current items; ground truth uses only *operating* items); `00299` correct segment (Corporate) but reported +$254M instead of −$473M (model dropped the negative sign).
- **Hallucinated:** `00080` PayPal WC (wrong sign, fabricated $63.8B / $83.8B figures wrapped in convincing payment-float reasoning); `00283` Pfizer–Upjohn spin-off cost ($12B vs actual $77.78M — 150× off, zero hedging). These are the most dangerous failures — authoritative tone, wrong facts.

**3. Are there patterns by *question_type*? Do some types fail more than others?**  
- **domain-relevant (5):** 1 correct / 3 partial / 1 wrong.
- **novel-generated (5):** 1 correct / 2 partial / 1 wrong / 1 refused.

The model succeeds when the answer needs *concepts* or *widely-known facts* and fails when it needs *exact figures from a specific filing*. This is exactly the gap RAG should close — expect the largest improvement on novel-generated questions and on the working-capital triad (00005/00070/00080), where without the filing the model can't apply the narrower *operating* WC definition.

---
## Task 2 – RAG reminder (5 points)

### 1. Indexing (documents → chunk + embed → vector store)

**How does it contribute to the pipeline?**  

Indexing is the most important part, it is the "ground truth" of how we interpreted meaning and it will affect all future search. it includes chunking which splits documents into pieces, embedding converts text to vectors for semantic similarity, and the vector store enables fast lookup. since it is done once, may take a lot of time, and need to be optimized, improving it is much more complex. 

**Where can it fail, concrete examples?**  
Rare domain terms (e.g. "EBITDAR", "MREL") get weak embeddings that don't cluster with their true semantic neighbors. It can also fail on close meanings (two phrasings of the same concept landing far apart), and on stale data if the index isn't rebuilt after a document update.
Chunking can fail in overlaps for example a financial table split across two chunks loses its row context; a term like "EBITDAR" may be poorly embedded by a general-purpose model; a stale index after a document update.

**Once or per query?**  

It is done once and is the base for all search and retrieval this is why it is so important.

### 2. Retrieval (user query → retrieval)

**How does it contribute to the pipeline?**  

Retrieval is getting the data, it is also critical since it should bring the right amount of documents not to overload the context with irrelevant data but enough to give a full answer. the quality and relevance of the data fetched is critical if this fails the generation would fail.
**Where can it fail, concrete examples?**  
Retrieval fails when it fetches from the wrong document, returns wrong-k results, or
scores irrelevant chunks higher. When not bringing enough answers, for example when calculating capital gain, giving partial information then sums calculation would be wrong, giving irrelevant data.  

**Once or per query?**  

Per query.

### 3. Generation (generation)

**How does it contribute to the pipeline?**  

Generation converts retrieved chunks + query into a coherent answer for the user;  Depending on the data retrieved, llm should compose a reasonable answer. relevant if it had the data and saying it doesn't know if it doesn't have it.

**Where can it fail?**  

It can miss some of the data, it can add irrelevant other data. without strong instruction it may hallucinate beyond the context or ignore retrieved facts. For example the model may attribute a number to the wrong company when two filings are in context; it can ignore a retrieved passage saying "not applicable" and  hallucinates a figure anyway.      
The model should use the correct information from the chunks, know when there is not enough information. Even of all the data is there, it might not use it correctly. it should, given the relevant data, compose the correct answer. 

**Once or per query?**

Per query.

---
## Task 3 – embed documents (15 points)

Downloading PDFs, splitting embedding and persisting a FAISS index to disk (`faiss_chunk1000/`) for reuse across Tasks 4–7.

In [9]:
# Download the PDFs for every doc_name in the filtered dataset.
# Cached under ./pdfs/ so re-running is a no-op.
#
# NOTE: pypdf needs `cryptography` (or `pycryptodome`) to decrypt AES-encrypted filings.
# If you see "DependencyError: cryptography>=3.1 is required for AES algorithm" after
# running this cell once, restart the kernel (Kernel → Restart) and re-run — pypdf
# caches the missing-crypto state from an earlier import.
import sys
!{sys.executable} -m pip install -q --upgrade pypdf
!{sys.executable} -m pip install -q cryptography pycryptodome

from pathlib import Path
import urllib.request
import urllib.error

PDF_DIR = Path("pdfs")
PDF_DIR.mkdir(exist_ok=True)
# github.com/.../tree/main/pdfs URL is the browsable directory page — it returns the HTML of GitHub's file
# listing UI, not the PDF bytes. raw.githubusercontent.com/patronus-ai/financebench/main/pdfs/<name>.pdf is the
#direct raw download URL 
GITHUB_PDF_BASE = "https://raw.githubusercontent.com/patronus-ai/financebench/main/pdfs"

needed_docs = sorted(df["doc_name"].unique())
print(f"Need {len(needed_docs)} PDFs")

missing = []
for name in needed_docs:
    dest = PDF_DIR / f"{name}.pdf"
    if dest.exists() and dest.stat().st_size > 0:
        continue
    url = f"{GITHUB_PDF_BASE}/{name}.pdf"
    try:
        urllib.request.urlretrieve(url, dest)
    except urllib.error.HTTPError as e:
        missing.append((name, e.code))
        if dest.exists():
            dest.unlink()

have = sum(1 for n in needed_docs if (PDF_DIR / f"{n}.pdf").exists())
print(f"Downloaded/cached: {have}/{len(needed_docs)}")
if missing:
    print(f"Missing (HTTP errors): {missing}")


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Need 42 PDFs
Downloaded/cached: 42/42


In [10]:
# Load each PDF page-by-page with PyPDFLoader and attach standardized metadata.
# PyPDFLoader's default `page` metadata is already 0-indexed → matches evidence_page_num.
from langchain_community.document_loaders import PyPDFLoader

docs_per_page = []
for name in needed_docs:
    path = PDF_DIR / f"{name}.pdf"
    if not path.exists():
        continue
    meta = doc_meta[name]
    pages = PyPDFLoader(str(path)).load()   # one Document per page
    for p in pages:
        page_num = p.metadata.get("page", 0)
        p.metadata = {
            "doc_name":    name,
            "company":     meta["company"],
            "doc_period":  meta["doc_period"],
            "page_number": int(page_num),
        }
        docs_per_page.append(p)

print(f"Loaded {len(docs_per_page)} page-level Documents from {len(needed_docs)} PDFs")
print("Sample metadata:", docs_per_page[0].metadata)

Loaded 5448 page-level Documents from 42 PDFs
Sample metadata: {'doc_name': '3M_2022_10K', 'company': '3M', 'doc_period': 2022, 'page_number': 0}


In [11]:
# Split into chunks; LangChain propagates parent page metadata to each chunk automatically.
# In recent langchain versions RecursiveCharacterTextSplitter lives in its own package.
import sys
!{sys.executable} -m pip install -q langchain-text-splitters

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(docs_per_page)

print(f"Split {len(docs_per_page)} pages → {len(chunks)} chunks")
print("Sample chunk metadata:", chunks[0].metadata)
print("Sample chunk text (first 200 chars):")
print(chunks[0].page_content[:200])


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Split 5448 pages → 24218 chunks
Sample chunk metadata: {'doc_name': '3M_2022_10K', 'company': '3M', 'doc_period': 2022, 'page_number': 0}
Sample chunk text (first 200 chars):
T able of Contents
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 
10-K
 
 
ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the f


In [12]:
# Embed with BAAI/bge-small-en-v1.5 and build/load the FAISS index on disk.
# Folder name includes the chunk size so Task 7 can add `faiss_chunk500`, `faiss_chunk2000`, etc.
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

FAISS_DIR = "faiss_chunk1000"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

if Path(FAISS_DIR).exists():
    vectorstore = FAISS.load_local(FAISS_DIR, embeddings, allow_dangerous_deserialization=True)
    print(f"Loaded existing FAISS index from {FAISS_DIR}")
else:
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(FAISS_DIR)
    print(f"Built and saved FAISS index → {FAISS_DIR}")

print(f"Total vectors in store: {vectorstore.index.ntotal}")

/var/folders/lg/vvqm3z4111l_5k_q7hn8kcmc0000gn/T/ipykernel_48557/638132544.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14910.88it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Built and saved FAISS index → faiss_chunk1000
Total vectors in store: 24218


In [13]:
# Retrieval sanity check on 3 questions: right doc? right page? evidence text visible in chunks?
#
# Evidence-text visibility is measured by word 8-gram overlap between each chunk and
# the union of the question's evidence_texts. Any shared 8-gram ⇒ evidence text is
# verbatim-present in that chunk.  (Shorter evidence texts fall back to smaller n.)
# The original verbose printout is preserved below for manual inspection.

SAMPLE_IDS = ["financebench_id_00005", "financebench_id_00080", "financebench_id_00206"]
K = 5

def _shingles(text, n=8):
    words = str(text).lower().split()
    n = min(n, max(1, len(words) - 1))
    return {" ".join(words[i:i + n]) for i in range(len(words) - n + 1)}

def evidence_overlap(chunk_text, evidence_texts, n=8):
    """Count of shared n-gram shingles, and total evidence shingles (for normalization)."""
    chunk_shings = _shingles(chunk_text, n=n)
    ev_shings = set()
    for ev in evidence_texts:
        ev_shings |= _shingles(ev, n=n)
    return len(chunk_shings & ev_shings), len(ev_shings)

sample_df = df[df["financebench_id"].isin(SAMPLE_IDS)].copy()
md_lines = ["# Task 3 – Retrieval Sanity Check (3 Questions)", ""]

for _, row in sample_df.iterrows():
    q              = row["question"]
    expected_doc   = row["doc_name"]
    expected_pages = row["evidence_page_nums"]
    expected_texts = row["evidence_texts"]

    retrieved = vectorstore.similarity_search(q, k=K)

    doc_hits = sum(1 for c in retrieved if c.metadata["doc_name"] == expected_doc)
    page_hit = any(
        c.metadata["doc_name"] == expected_doc
        and c.metadata["page_number"] in expected_pages
        for c in retrieved
    )
    text_hit = any(
        evidence_overlap(c.page_content, expected_texts)[0] > 0 for c in retrieved
    )

    # ── console: verbose, manual-inspection friendly ─────────────────────────
    print("=" * 95)
    print(f"[{row['financebench_id']}]  {q[:110]}")
    print(f"Expected doc: {expected_doc}  |  evidence pages: {expected_pages}")
    print(f"Chunks from correct doc: {doc_hits}/{K}   |   Any chunk on evidence page: {page_hit}   |   Evidence text visible (8-gram): {text_hit}")
    for i, c in enumerate(retrieved, 1):
        page_ok = (c.metadata["doc_name"] == expected_doc
                   and c.metadata["page_number"] in expected_pages)
        ov_hits, ov_total = evidence_overlap(c.page_content, expected_texts)
        marker = "✓" if page_ok else " "
        text_marker = "★" if ov_hits > 0 else " "
        snippet = c.page_content[:120].replace("\n", " ")
        print(f"  {marker}{text_marker} rank {i}  {c.metadata['doc_name']}  p{c.metadata['page_number']}  "
              f"[ev 8-grams: {ov_hits}/{ov_total}]  | {snippet}...")
    if expected_texts:
        print(f"  Evidence text (truncated): {expected_texts[0][:220]}...")

    # ── markdown: structured capture for task3_observe_3docs.md ──────────────
    md_lines += [
        f"## {row['financebench_id']}",
        "",
        f"**Question:** {q}",
        "",
        f"- Expected doc: `{expected_doc}`",
        f"- Evidence pages: `{expected_pages}`",
        f"- Chunks from correct doc: **{doc_hits}/{K}**",
        f"- Any chunk on an evidence page: **{page_hit}**",
        f"- Evidence text visible (8-gram overlap): **{text_hit}**",
        "",
        "| rank | doc | page | page_match | ev_8grams | chunk (first 160 chars) |",
        "|------|-----|------|------------|-----------|--------------------------|",
    ]
    for i, c in enumerate(retrieved, 1):
        page_ok = (c.metadata["doc_name"] == expected_doc
                   and c.metadata["page_number"] in expected_pages)
        ov_hits, ov_total = evidence_overlap(c.page_content, expected_texts)
        snippet = c.page_content[:160].replace("\n", " ").replace("|", "\\|")
        md_lines.append(
            f"| {i} | `{c.metadata['doc_name']}` | {c.metadata['page_number']} | "
            f"{'✓' if page_ok else ' '} | {ov_hits}/{ov_total} | {snippet}... |"
        )
    md_lines += [""]
    for j, ev in enumerate(expected_texts):
        md_lines += [f"**Evidence text [{j}]** (page {expected_pages[j] if j < len(expected_pages) else '?'}):", "", "> " + str(ev).replace("\n", " "), ""]

out_path = Path("task3_observe_3docs.md")
out_path.write_text("\n".join(md_lines), encoding="utf-8")
print(f"\nMarkdown report written → {out_path}")

[financebench_id_00005]  Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevan
Expected doc: CORNING_2022_10K  |  evidence pages: [59]
Chunks from correct doc: 3/5   |   Any chunk on evidence page: False   |   Evidence text visible (8-gram): False
     rank 1  CORNING_2022_10K  p101  [ev 8-grams: 0/171]  | (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Heml...
     rank 2  CORNING_2022_10K  p102  [ev 8-grams: 0/171]  | (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Heml...
     rank 3  3M_2022_10K  p37  [ev 8-grams: 0/171]  | not defined under U.S. generally accepted accounting principles and may not be computed the same as similarly titled mea...
     rank 4  3M_2023Q2_10Q  p70  [ev 8-grams: 0/171]  | Current assets $ 15,754 $ 14,688 $ 1,066  Less: Current liabilities 10,936 9,523 1,413  Worki

### Task 3 – Observations

**Right document (doc_name match)?**  
For Corning(id_00005), 3/5 right docs. ranks 1-2, 5 were the right documents, rank 3-4 were another company(3M).
For PayPal(id_00080) all 5 ranks were from correct doc.
For JPM, none of the ranks were from the correct doc rank2 was the only one from JPMORGAN_2022Q2_10Q instead of JPMORGAN_2022_10K.

Note the doc-recall vs page-recall tradeoff: PayPal hit 5/5 on the right *document* but still 0/5 on the right *page* — finding the right filing isn't enough if the retriever lands on MD&A pages instead of the balance sheet.

**Evidence text visible in retrieved chunks?**  
trying to find 8-grams (a contiguous sequence of 8 words, in order) in retrieved text in all 3 companies resulted in zero match.

**Right page (chunk page_number vs evidence_page_num)?**  
In all the companies retrieved chunks were not found in evidence page.
For Corning and PayPal retrieval lands on narrative/MD&A pages that
talk about working capital, not the balance-sheet table that holds the
numbers. For JPM it misses the 10-K entirely because "gross margins"
isn't vocabulary JPM's filing uses. A reranker or hybrid retriever might help later.

To summarize results looks pretty poor, in total 8/15 documents were correct, no text in evidence text, no page number that matches evidence pages number. This may be since as explained FinanceBench is a hard dataset and BGE-small is a general-purpose embedder.

---
## Task 4 – RAG pipeline (25 points)

A single function `answer_with_rag(query, k=4)` that:
1. Embeds the query with the same `BAAI/bge-small-en-v1.5` model (via the FAISS wrapper built in Task 3).
2. Retrieves the top-k chunks from `vectorstore`.
3. Formats them into a prompt block with clear separators and the `doc_name` / `page_number` of each chunk — so the generator can cite its source.
4. If retrieval is empty, the prompt tells the model there is no context instead of leaving an empty block.
5. Sends the prompt (+ a strict system prompt) to `Llama-3.3-70B-Instruct` and returns `{"answer": str, "retrieved_chunks": list}`.

The **system prompt** instructs the model to answer *only* from the provided context, to say so explicitly when the context does not contain the answer (rather than guessing), to keep answers concise, and to cite the `doc_name` for each fact.

In [14]:
# ── Task 4: RAG pipeline ─────────────────────────────────────────────────────

RAG_SYSTEM_PROMPT = (
    "Follow these rules strictly:\n"
    "1. Answer ONLY from the provided context below. Do not use any outside knowledge.\n"
    "2. If the context does not contain the answer, say so explicitly — reply exactly: "
    "\"The provided context does not contain the answer.\" Do not guess.\n"
    "3. Keep answers concise — one short paragraph at most.\n"
    "4. Cite the source for each fact by appending the doc_name in parentheses, "
    "e.g. \"(CORNING_2022_10K)\". If a fact comes from multiple filings, cite each.\n"
)

NO_CONTEXT_BLOCK = "[NO CONTEXT RETRIEVED — the retriever returned zero chunks for this query.]"


def _format_context(chunks) -> str:
    """Stitch retrieved chunks into a single context block with clear separators + metadata."""
    if not chunks:
        return NO_CONTEXT_BLOCK
    parts = []
    for i, c in enumerate(chunks, 1):
        doc = c.metadata.get("doc_name", "UNKNOWN")
        page = c.metadata.get("page_number", "?")
        parts.append(
            f"[Chunk {i} | doc_name={doc} | page_number={page}]\n"
            f"{c.page_content.strip()}"
        )
    return "\n\n---\n\n".join(parts)


def answer_with_rag(query: str, k: int = 4) -> dict:
    """Retrieve top-k chunks from the FAISS store and generate a grounded answer.

    Returns
    -------
    dict with keys:
      - answer: str, the generation model's final answer.
      - retrieved_chunks: list of dicts {doc_name, page_number, content}.
    """
    retrieved = vectorstore.similarity_search(query, k=k)

    context = _format_context(retrieved)
    user_message = (
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        "Answer using ONLY the context above. Cite doc_name for each fact."
    )

    answer = llm_generate(user_message, system_prompt=RAG_SYSTEM_PROMPT)

    retrieved_chunks = [
        {
            "doc_name":    c.metadata.get("doc_name"),
            "page_number": c.metadata.get("page_number"),
            "content":     c.page_content,
        }
        for c in retrieved
    ]

    return {"answer": answer, "retrieved_chunks": retrieved_chunks}


print("answer_with_rag defined.")
print("System prompt preview:\n")
print(RAG_SYSTEM_PROMPT)

In [15]:
# ── Quick demo: run 3 sample queries through the pipeline and dump a detailed
# trace to task4_generation.md (summary stays in the notebook, full context +
# chunk text live in the markdown file so they're easy to scan).

SAMPLE_IDS_T4 = ["financebench_id_00005", "financebench_id_00080", "financebench_id_00206"]
DEMO_K = 4

demo_rows = df[df["financebench_id"].isin(SAMPLE_IDS_T4)].copy()

md_lines = ["# Task 4 – RAG pipeline: detailed demo trace", "",
            f"k = {DEMO_K}, embedding = `{EMBED_MODEL_NAME}`, generation = `{LLM_MODEL}`", ""]

first = True
for _, row in demo_rows.iterrows():
    q         = row["question"]
    fb_id     = row["financebench_id"]
    gt        = row["ground_truth"]
    exp_doc   = row["doc_name"]
    exp_pages = row["evidence_page_nums"]

    result = answer_with_rag(q, k=DEMO_K)

    # console: compact summary
    print("=" * 95)
    print(f"[{fb_id}]  {q[:110]}")
    print(f"Expected doc: {exp_doc}  |  evidence pages: {exp_pages}")

    # On the first example, show the dict structure explicitly so it's clear what
    # the function returns (assignment requires answer + retrieved_chunks with
    # doc_name and page_number metadata).
    if first:
        print(f"\nReturned dict keys: {list(result.keys())}")
        print(f"Each chunk's keys : {list(result['retrieved_chunks'][0].keys())}")
        first = False

    print(f"\nRAG answer:\n{result['answer']}\n")
    print(f"Ground truth:\n{gt[:250]}")
    print("\nRetrieved chunks (from result['retrieved_chunks']):")
    for i, ch in enumerate(result["retrieved_chunks"], 1):
        match = "✓" if ch["doc_name"] == exp_doc and ch["page_number"] in exp_pages else " "
        snippet = ch["content"].strip().replace("\n", " ")[:140]
        print(f"  {match} {i}. doc_name={ch['doc_name']}  page_number={ch['page_number']}")
        print(f"        {snippet}...")

    # markdown: full trace
    md_lines += [
        f"## {fb_id}", "",
        f"**Question:** {q}", "",
        f"- Expected doc: `{exp_doc}`",
        f"- Evidence pages: `{exp_pages}`", "",
        "### RAG answer", "",
        "```", result["answer"], "```", "",
        "### Ground truth", "",
        "> " + str(gt).replace("\n", " "), "",
        "### Retrieved chunks", "",
    ]
    for i, ch in enumerate(result["retrieved_chunks"], 1):
        match = "✓" if ch["doc_name"] == exp_doc and ch["page_number"] in exp_pages else " "
        md_lines += [
            f"**Chunk {i}** {match} — `{ch['doc_name']}` page **{ch['page_number']}**",
            "",
            "```",
            ch["content"].strip(),
            "```",
            "",
        ]
    md_lines.append("---")
    md_lines.append("")

# Also demonstrate the empty-retrieval safeguard by asking something with no
# financial content — retriever still returns k chunks, but we can verify the
# prompt formatting branch on an empty list directly.
md_lines += [
    "## Empty-retrieval branch (sanity check)", "",
    "Calling `_format_context([])` should produce the explicit no-context marker used in the prompt, "
    "so the model is told there is nothing rather than being handed a blank block.",
    "",
    "```",
    _format_context([]),
    "```",
    "",
]

out_path = Path("task4_generation.md")
out_path.write_text("\n".join(md_lines), encoding="utf-8")
print(f"\nDetailed trace written → {out_path}")

[financebench_id_00005]  Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevan
Expected doc: CORNING_2022_10K  |  evidence pages: [59]

Returned dict keys: ['answer', 'retrieved_chunks']
Each chunk's keys : ['doc_name', 'page_number', 'content']

RAG answer:
The provided context does not contain the answer. The context only provides information about Corning's financial statements and transactions (CORNING_2022_10K), but it does not mention working capital. The working capital information is only mentioned in the context of 3M's financial statements (3M_2022_10K, 3M_2023Q2_10Q).

Ground truth:
Yes. Corning had a positive working capital amount of $831 million by FY 2022 close. This answer considers only operating current assets and current liabilities that were clearly shown in the balance sheet.

Retrieved chunks (from result['retrieved_chunks']):
    1. doc_name=CORNING_2022_10K  page_number=101
        (1) Corning obtained a c

### Task 4 – Insights

**Bottom line:** for all three demo questions the model replied "The provided context does not contain the answer." Whether that's a win depends on what the naive baseline would have done — the same refusal can be a *save* (over a hallucination), a *wash* (over a partial guess), or an outright *loss* (over a correct answer).

**PayPal (`00080`) — RAG helped.** Naive confidently produced a *negative* working capital with fabricated balance-sheet numbers (ground truth is positive). Retrieval also missed the balance-sheet page, but the strict prompt forced a refusal instead of a wrong confident answer — clearly the safer failure.

**Corning (`00005`) — borderline.** Naive said positive WC (~$2.9B), which matches the GT *direction* but uses *all* current items rather than the *operating* subset the ground truth defines, so the figure is off (GT $831M). Since the naive rationale isn't grounded in real figures, we can't tell whether the right direction came from real knowledge or coincidence. RAG refusing is neither clearly worse nor clearly better.

**JPM (`00206`) — RAG hurt.** Naive correctly said gross margin isn't a relevant metric for a bank — almost word-for-word the ground-truth answer, drawn from memorized industry knowledge. Retrieval missed `JPMORGAN_2022_10K` and surfaced BestBuy/AMD/Boeing chunks, so RAG refused. A correct answer turned into a refusal because retrieval failed to surface the right filing.

---
## Task 5 – run and compare (10 points)

Run the same 10 questions from Task 1 through `answer_with_rag` and put the naive answer and the RAG answer side by side with the ground truth.

In [16]:
# Run answer_with_rag on each of the 10 Task 1 questions; reuse the naive
# answers already stored in task1_df.
#
# Self-contained safety net: if the kernel was restarted and task1_df is gone,
# rehydrate it from the Task 1 xlsx so we don't have to re-call the LLM for
# the naive answers. (Task 3's `vectorstore` and Task 4's `answer_with_rag`
# must still be in the kernel — re-run those cells if you see NameError.)
if "task1_df" not in globals():
    task1_df = pd.read_excel("assignment2_naive_generation_ilay_nisim_boris_kogan.xlsx")
    print("Reloaded task1_df from Task 1 xlsx.")

RAG_K = 4

rag_answers = []
retrieved_logs = []  # kept for inspection / debugging; not written to the xlsx
for _, row in task1_df.iterrows():
    print(f"[{row['financebench_id']}] querying RAG...")
    result = answer_with_rag(row["question"], k=RAG_K)
    rag_answers.append(result["answer"])
    retrieved_logs.append(result["retrieved_chunks"])

task5_df = task1_df[["financebench_id", "question_type", "question",
                     "naive_answer", "ground_truth"]].copy()
task5_df["RAG_answer"] = rag_answers
task5_df = task5_df[["financebench_id", "question_type", "question",
                     "naive_answer", "RAG_answer", "ground_truth"]]

print("\nDone. Side-by-side preview:")
for _, row in task5_df.iterrows():
    print("=" * 90)
    print(f"[{row['financebench_id']}] ({row['question_type']}) {row['question'][:110]}")
    print(f"  Naive: {row['naive_answer'][:160]}")
    print(f"  RAG  : {row['RAG_answer'][:160]}")
    print(f"  GT   : {str(row['ground_truth'])[:160]}")

# Persist retrieved chunks so the Task 5 discussion can be audited without
# re-running retrieval (e.g. the 00299 claim that p4 was surfaced).
with open("task5_retrieved_chunks.json", "w") as _f:
    json.dump(
        {fb: chunks for fb, chunks in zip(task1_df["financebench_id"], retrieved_logs)},
        _f, indent=2,
    )

[financebench_id_00005] querying RAG...
[financebench_id_00070] querying RAG...
[financebench_id_00080] querying RAG...
[financebench_id_00206] querying RAG...
[financebench_id_00215] querying RAG...
[financebench_id_00283] querying RAG...
[financebench_id_00288] querying RAG...
[financebench_id_00299] querying RAG...
[financebench_id_00302] querying RAG...
[financebench_id_00382] querying RAG...

Done. Side-by-side preview:
[financebench_id_00005] (domain-relevant) Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevan
  Naive: Based on Corning's FY2022 data, the company's current assets were $9.35 billion and current liabilities were $6.44 billion. This results in a positive working c
  RAG  : The provided context does not contain the answer. The context only provides information about Corning's financial statements, such as depreciation expense, rese
  GT   : Yes. Corning had a positive working capital amount of $831 million by

In [17]:
OUT_FILE = "assignment2_run_and_compare_ilay_nisim_boris_kogan.xlsx"
task5_df.to_excel(OUT_FILE, index=False)
print(f"Saved → {OUT_FILE}")

Saved → assignment2_run_and_compare_ilay_nisim_boris_kogan.xlsx


### Task 5 – Discussion

**Caveat on "naive correct".** Several naive answers got the *direction* right while making up numbers (`00005`, `00070`, `00302`). When the question only requires a one-bit answer (yes/no, positive/negative, grew/shrunk) and the rationale is built on fabricated figures, we can't tell whether the right direction came from real knowledge or a lucky coin flip. The cases below make that distinction explicit.

**1. Did RAG help?**
- `00080` **PayPal working capital** — naive confidently produced a *negative* working capital with fabricated current-asset/liability figures (GT positive ~$1.6Bn). Retrieval also missed the balance-sheet page, but the strict prompt forced a refusal instead of a wrong confident answer.
- `00288` **BestBuy cash drop FY23 → Q2 FY24** — the clearest win. Naive refused because the period is past the model's training cutoff. RAG retrieved both `BESTBUY_2023_10K` and `BESTBUY_2024Q2_10Q` and produced a grounded, cited answer that matches the GT direction. Exactly what RAG is for.
- `00382` **MGM highest-EBITDAR region** — both pointed to Las Vegas, but RAG grounded the answer in `MGMRESORTS_2022_10K` with an Adjusted Property EBITDAR figure and a citation. Same verdict, stronger provenance.

**2. Did RAG hurt?**
- `00206` **JPM gross margin** — the only clear "fully-correct → refusal" in the set. Naive matched the GT almost word-for-word ("not a relevant metric for a bank") from memorized industry knowledge; retrieval missed `JPMORGAN_2022_10K` and surfaced BestBuy / AMD / Boeing chunks, so RAG refused.
- `00070` **American Water Works WC** — *partially* correct naive: direction right (negative), but the number was off by an order of magnitude (`-$120M` vs GT `-$1561M`). Since the rationale didn't reference real figures, the right direction may not be knowledge at all. Retrieval landed in the right filing but wrong pages and RAG refused — we're trading a possibly-coincidental partial answer for an honest refusal.
- `00302` **Pfizer PPNE FY20→FY21** — *partially* correct naive: said "yes, PPNE grew" with no figures, matching the GT direction. The naive answer even mislabels PPNE as "Pharmaceutical Pipeline, New Products, and Exclusions" (the financial term is Property, Plant & Equipment, Net) — so the *correct* yes/no is essentially a coin flip dressed up as analysis. RAG retrieved from `PFIZER_2021_10K` and still refused.
- `00283` **Pfizer–Upjohn spin-off cost** — naive hallucinated `$12B`, RAG cited `$1.6B` from `PFIZER_2021_10K`; GT is `$77.78M` (evidence in `Pfizer_2023Q2_10Q`). Both wrong with confidence — RAG just adds a misleading citation that makes the wrong number look provenanced.
- `00299` **JPM lowest-revenue segment Q1 2021** — retrieval did surface `JPMORGAN_2021Q1_10Q` p4, but the model still refused. The chunk likely contains the segment numbers as a table without the explicit "Corporate had the lowest net revenue" phrasing, and the strict prompt blocks the model from doing the comparison itself. **Pulling neighboring chunks** (e.g. larger overlap, or fetching the chunk before/after each hit) might surface the surrounding narrative that names the segment — a concrete Task 7 experiment.

**3. Patterns by `question_type`.**  
On this 10-question slice the split is roughly symmetric: **domain-relevant 1 help / 3 hurt / 1 neutral**, **novel-generated 2 help / 3 hurt**. The interesting pattern isn't *type* but *what kind of knowledge the question relies on*:
- **RAG wins** when the answer needs a specific recent figure the model couldn't have memorized (`00288` BestBuy Q2 FY24 is the cleanest case) — genuinely novel, filing-specific facts.
- **RAG loses** when the naive answer came from broad industry knowledge (`00206` JPM is the clearest example) — the strict prompt turns a correct-from-memory answer into a refusal unless retrieval surfaces the *exact* evidence wording.

A real picture will only emerge on the full 100 questions in Task 6. Mechanically, the bottleneck on this slice is clearly **retrieval**: 4 of the 6 hurt/neutral cases retrieved *the right filing* but the wrong pages, so the generator had no choice but to refuse. That motivates the retrieval-side experiments in Task 7 (larger k, reranker, chunk-size sweep).

**Note on the assignment's "small vector store" debugging tip.** A "1-to-2-doc index" means *rebuilding* FAISS from scratch with **only one or two entire PDFs** (all pages, all chunks of those PDFs) — not 1–2 pages and not larger chunks. Stripping out the noise from 40+ unrelated filings would let us see, for a query like the JPM gross-margin one, *which page* of `JPMORGAN_2022_10K` the embedding model actually surfaces — which would tell us whether the miss is a chunk-granularity issue (wrong page within the right doc) or a vocabulary mismatch (the phrase "gross margin" simply doesn't appear in a bank's 10-K).

### Task 5 – Debug: small vector store sanity check

Independent follow-up to the note at the end of the Task 5 discussion. We rebuild a **mini FAISS index from a single PDF** (`JPMORGAN_2022_10K`) — same embedding model, same chunking — and re-run the JPM gross-margin query on it. With 40+ unrelated filings stripped out, whatever comes back *must* be the best match inside the correct 10-K. That tells us whether the Task 3 / Task 5 miss was:

- a **chunk-granularity** issue (wrong page within the right doc), or
- a **vocabulary mismatch** (the phrase "gross margin" simply doesn't appear in a bank's 10-K).

We also grep the raw PDF pages for the literal phrase "gross margin" as a tiebreaker. All variables are prefixed `small_` and nothing from Tasks 1–5 is reused or mutated.

In [18]:
# ── Task 5 debug: independent small vector store ─────────────────────────────
# Mini FAISS index over ONE PDF (JPMORGAN_2022_10K), fresh variable names only.
# Does NOT touch df, vectorstore, embeddings, answer_with_rag, or any earlier var.

from pathlib import Path as _SmallPath
from langchain_community.document_loaders import PyPDFLoader as _SmallPyPDFLoader
from langchain_community.vectorstores import FAISS as _SmallFAISS
from langchain_community.embeddings import HuggingFaceEmbeddings as _SmallHFE
from langchain_text_splitters import RecursiveCharacterTextSplitter as _SmallSplitter

SMALL_DOC = "JPMORGAN_2022_10K"
SMALL_QUERY = (
    "Are JPM's gross margins historically consistent (not fluctuating more than "
    "roughly 2% each year)? If gross margins are not a relevant metric for a "
    "company like this, then please state that and explain why."
)
SMALL_EXPECTED_PAGES = [2]   # financebench_id_00206 evidence page
SMALL_K = 8

small_pdf = _SmallPath("pdfs") / f"{SMALL_DOC}.pdf"
assert small_pdf.exists(), f"PDF not found: {small_pdf}"

small_pages = _SmallPyPDFLoader(str(small_pdf)).load()
for _p in small_pages:
    _p.metadata = {
        "doc_name":    SMALL_DOC,
        "page_number": int(_p.metadata.get("page", 0)),
    }

small_chunks = _SmallSplitter(
    chunk_size=1000, chunk_overlap=150
).split_documents(small_pages)

small_emb = _SmallHFE(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
small_vs = _SmallFAISS.from_documents(small_chunks, small_emb)

print(f"Mini index built from {SMALL_DOC}: "
      f"{len(small_pages)} pages → {len(small_chunks)} chunks "
      f"({small_vs.index.ntotal} vectors)")
print()

# ── retrieval with scores ────────────────────────────────────────────────────
small_hits = small_vs.similarity_search_with_score(SMALL_QUERY, k=SMALL_K)

print(f"Query: {SMALL_QUERY}")
print(f"Evidence page from FinanceBench: {SMALL_EXPECTED_PAGES}")
print(f"Top-{SMALL_K} retrieved (smaller distance = closer):")
for _i, (_ch, _score) in enumerate(small_hits, 1):
    _p = _ch.metadata["page_number"]
    _mark = "✓" if _p in SMALL_EXPECTED_PAGES else " "
    _snip = _ch.page_content.replace("\n", " ").strip()[:140]
    print(f"  {_mark} rank {_i:>2}  page {_p:>3}  dist={_score:.3f}  | {_snip}...")

# ── vocabulary tiebreaker: does "gross margin" appear anywhere? ─────────────
small_gm_pages = [
    _p.metadata["page_number"]
    for _p in small_pages
    if "gross margin" in _p.page_content.lower()
]
print()
print(f"Pages in {SMALL_DOC} literally containing 'gross margin': "
      f"{small_gm_pages if small_gm_pages else 'NONE'}")

# ── interpretation hint ──────────────────────────────────────────────────────
_top_page = small_hits[0][0].metadata["page_number"]
if _top_page in SMALL_EXPECTED_PAGES:
    print("\n→ Granularity issue was NOT the bottleneck: "
          "mini index surfaces the evidence page at rank 1.")
elif not small_gm_pages:
    print("\n→ Vocabulary mismatch: 'gross margin' does not appear in the filing, "
          "so BGE-small cannot anchor the query to the evidence page even with no noise.")
else:
    print("\n→ Mixed: 'gross margin' exists in the filing but the mini index still "
          "misses the evidence page — suggests chunk-granularity + embedding weakness.")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15589.30it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Mini index built from JPMORGAN_2022_10K: 382 pages → 1965 chunks (1965 vectors)

Query: Are JPM's gross margins historically consistent (not fluctuating more than roughly 2% each year)? If gross margins are not a relevant metric for a company like this, then please state that and explain why.
Evidence page from FinanceBench: [2]
Top-8 retrieved (smaller distance = closer):
    rank  1  page 208  dist=0.537  | Total $ (123,728) $ (381,802) $ (37,192) $ (542,722) $ 3,591 $ (4,704) $ (1,113)  December 31, 2021 (in millions) <1 year 1–5 years >5 years...
    rank  2  page  28  dist=0.552  | not occur. Any such lapses or deficiencies could result in  inaccurate financial reporting which, in turn, could: • materially and adversely...
    rank  3  page  27  dist=0.553  | processes could result in ineffective risk management  practices. These deficiencies could also result in inaccurate  or untimely risk repor...
    rank  4  page  78  dist=0.558  | data may have been different if all share cl

**Conclusion.** Even with all other filings removed, the mini index never surfaces the evidence page (top-8 distances ~0.54–0.57 across unrelated pages), and the phrase "gross margin" appears on **zero** pages of `JPMORGAN_2022_10K`. The Task 5 failure on `00206` is a **vocabulary mismatch**, not a chunk-granularity issue — no amount of reranking or larger `k` would have rescued it.

---
## Task 6 – evaluation (20 points)

Evaluate the pipeline on the full 100-question filtered dataset with three metrics:
- **Correctness**: LLM-as-judge (DeepSeek), 
- **Faithfulness**: on the first 20 examples.
- **Retrieval hit-rate**: page-hit@k for k ∈ {1, 3, 5}, on all 100.

The block below is **self-contained** — if you just opened the kernel, start running from here. It re-imports, re-loads the dataset, loads the FAISS index from `faiss_chunk1000/` (built in Task 3), and defines a local `rag_answer_t6(...)`. All intermediate results are cached to `task6_*.json` so re-runs after a crash / rate-limit only redo what's missing.

In [1]:
# ── Task 6 setup: standalone (does not depend on Task 1–5 kernel state) ──
import os, json, time
import pandas as pd
import numpy as np
from pathlib import Path
from openai import OpenAI
from datasets import load_dataset
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

load_dotenv()

T6_NEBIUS_KEY = os.environ["NEBIUS_API_KEY"]
T6_NEBIUS_BASE = "https://api.studio.nebius.ai/v1/"
T6_GEN_MODEL   = "meta-llama/Llama-3.3-70B-Instruct"
T6_JUDGE_MODEL = "deepseek-ai/DeepSeek-V3.2"   # per Task 6 spec
T6_EMBED_MODEL = "BAAI/bge-small-en-v1.5"
T6_FAISS_DIR   = "faiss_chunk1000"
T6_RAG_K       = 4
T6_PAGE_HIT_K  = [1, 3, 5]   # retrieve top-5 once, subset for @1/@3

t6_nebius = OpenAI(api_key=T6_NEBIUS_KEY, base_url=T6_NEBIUS_BASE)

# ── dataset: filter, rename, parse evidence pages, sort ──────────────────────
def _t6_parse_evidence(ev):
    if isinstance(ev, str):
        ev = json.loads(ev)
    if isinstance(ev, np.ndarray):
        ev = ev.tolist()
    if not isinstance(ev, list):
        ev = [ev]
    out = []
    for e in ev:
        if not isinstance(e, dict):
            e = dict(e)
        out.append(int(e.get("evidence_page_num", -1)))
    return out

t6_df = load_dataset("PatronusAI/financebench", split="train").to_pandas()
t6_df = t6_df[t6_df["question_type"] != "metrics-generated"].copy()
t6_df = t6_df.rename(columns={"answer": "ground_truth"})
t6_df["evidence_page_nums"] = t6_df["evidence"].apply(_t6_parse_evidence)
t6_df = t6_df.sort_values("financebench_id").reset_index(drop=True)

# ── FAISS (reuse the Task 3 index on disk) ───────────────────────────────────
t6_embed = HuggingFaceEmbeddings(
    model_name=T6_EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
t6_vs = FAISS.load_local(T6_FAISS_DIR, t6_embed, allow_dangerous_deserialization=True)

# ── local copy of answer_with_rag (same prompts / logic as Task 4) ───────────
T6_RAG_SYS = (
    "Follow these rules strictly:\n"
    "1. Answer ONLY from the provided context below. Do not use any outside knowledge.\n"
    "2. If the context does not contain the answer, say so explicitly — reply exactly: "
    "\"The provided context does not contain the answer.\" Do not guess.\n"
    "3. Keep answers concise — one short paragraph at most.\n"
    "4. Cite the source for each fact by appending the doc_name in parentheses.\n"
)

def rag_answer_t6(query, k=T6_RAG_K):
    chunks = t6_vs.similarity_search(query, k=k)
    if chunks:
        ctx = "\n\n---\n\n".join(
            f"[Chunk {i} | doc_name={c.metadata.get('doc_name')} | "
            f"page_number={c.metadata.get('page_number')}]\n{c.page_content.strip()}"
            for i, c in enumerate(chunks, 1)
        )
    else:
        ctx = "[NO CONTEXT RETRIEVED]"
    resp = t6_nebius.chat.completions.create(
        model=T6_GEN_MODEL,
        messages=[
            {"role": "system", "content": T6_RAG_SYS},
            {"role": "user",   "content": f"Context:\n{ctx}\n\nQuestion: {query}"},
        ],
        temperature=0.1,
        max_tokens=500,
    )
    return {
        "answer": resp.choices[0].message.content.strip(),
        "retrieved_chunks": [
            {"doc_name": c.metadata.get("doc_name"),
             "page_number": c.metadata.get("page_number"),
             "content": c.page_content}
            for c in chunks
        ],
    }

print(f"Task 6 setup OK. Dataset: {len(t6_df)} rows. "
      f"FAISS: {t6_vs.index.ntotal} vectors from {T6_FAISS_DIR}/.")

/Users/YOUR_USER/dev/cloud/nebius/lesson5_rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/lg/vvqm3z4111l_5k_q7hn8kcmc0000gn/T/ipykernel_53540/1543458106.py:47: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  t6_embed = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8432.93it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEX

Task 6 setup OK. Dataset: 100 rows. FAISS: 24218 vectors from faiss_chunk1000/.


In [2]:
# ── Generate RAG answer (k=4) + retrieve top-5 for page-hit. Cache to disk. ──
T6_RAG_CACHE = "task6_rag.json"

t6_rag = json.loads(Path(T6_RAG_CACHE).read_text()) if Path(T6_RAG_CACHE).exists() else {}
print(f"Starting with {len(t6_rag)}/{len(t6_df)} cached.")

for i, row in t6_df.iterrows():
    fb = row["financebench_id"]
    if fb in t6_rag:
        continue
    try:
        res = rag_answer_t6(row["question"], k=T6_RAG_K)
        top5 = t6_vs.similarity_search(row["question"], k=max(T6_PAGE_HIT_K))
        t6_rag[fb] = {
            "answer": res["answer"],
            "retrieved_chunks": res["retrieved_chunks"],
            "top5_pages": [
                {"doc_name": c.metadata.get("doc_name"),
                 "page_number": c.metadata.get("page_number")}
                for c in top5
            ],
        }
    except Exception as e:
        print(f"  [{fb}] ERROR: {e} — skipping, re-run cell to retry")
        continue
    if (i + 1) % 10 == 0:
        Path(T6_RAG_CACHE).write_text(json.dumps(t6_rag, indent=1))
        print(f"  {i+1}/{len(t6_df)} done")

Path(T6_RAG_CACHE).write_text(json.dumps(t6_rag, indent=1))
print(f"Done. RAG cache: {len(t6_rag)} entries → {T6_RAG_CACHE}")

Starting with 0/100 cached.
  10/100 done
  20/100 done
  30/100 done
  40/100 done
  50/100 done
  60/100 done
  70/100 done
  80/100 done
  90/100 done
  100/100 done
Done. RAG cache: 100 entries → task6_rag.json


In [3]:
# ── Correctness: DeepSeek judge, binary verdict + 1-sentence justification ──
T6_CORR_CACHE = "task6_correctness.json"

T6_JUDGE_SYS = (
    "You are a strict financial-QA judge. Given a question, a ground-truth answer, "
    "and a candidate answer, decide whether the candidate is correct.\n"
    "Return STRICTLY as JSON: "
    "{\"verdict\": \"correct\" | \"incorrect\", \"justification\": \"<one short sentence>\"}.\n"
    "Rules:\n"
    "- A candidate is \"correct\" only if it conveys the same factual claim as the "
    "ground truth (minor wording differences ok).\n"
    "- Numbers must match (small rounding ok); directions (positive/negative, grew/shrunk) "
    "must match.\n"
    "- If the candidate refuses, says it has no answer, or is off-topic, mark it \"incorrect\".\n"
    "- Use only the ground truth — do not apply outside knowledge."
)

def _t6_judge(question, gt, candidate):
    resp = t6_nebius.chat.completions.create(
        model=T6_JUDGE_MODEL,
        messages=[
            {"role": "system", "content": T6_JUDGE_SYS},
            {"role": "user",
             "content": f"Question: {question}\n\nGround truth:\n{gt}\n\nCandidate answer:\n{candidate}"},
        ],
        temperature=0.0,
        max_tokens=200,
    )
    raw = resp.choices[0].message.content.strip()
    # strip potential code fences
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.lower().startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    try:
        parsed = json.loads(raw)
        verdict = parsed.get("verdict", "incorrect").lower()
        if verdict not in ("correct", "incorrect"):
            verdict = "incorrect"
        return {"verdict": verdict, "justification": parsed.get("justification", "")}
    except Exception:
        # fallback: parse from text
        low = raw.lower()
        v = "correct" if ("\"correct\"" in low or low.startswith("correct")) and "incorrect" not in low else "incorrect"
        return {"verdict": v, "justification": raw[:200]}

t6_corr = json.loads(Path(T6_CORR_CACHE).read_text()) if Path(T6_CORR_CACHE).exists() else {}
print(f"Starting with {len(t6_corr)}/{len(t6_df)} cached.")

for i, row in t6_df.iterrows():
    fb = row["financebench_id"]
    if fb in t6_corr:
        continue
    if fb not in t6_rag:
        print(f"  [{fb}] no RAG answer yet — run the previous cell first")
        continue
    try:
        t6_corr[fb] = _t6_judge(row["question"], row["ground_truth"], t6_rag[fb]["answer"])
    except Exception as e:
        print(f"  [{fb}] judge error: {e}")
        continue
    if (i + 1) % 10 == 0:
        Path(T6_CORR_CACHE).write_text(json.dumps(t6_corr, indent=1))
        print(f"  {i+1}/{len(t6_df)} judged")

Path(T6_CORR_CACHE).write_text(json.dumps(t6_corr, indent=1))
correct_n = sum(1 for v in t6_corr.values() if v["verdict"] == "correct")
print(f"Done. Correctness: {correct_n}/{len(t6_corr)} correct → {T6_CORR_CACHE}")

Starting with 0/100 cached.
  10/100 judged
  20/100 judged
  30/100 judged
  40/100 judged
  50/100 judged
  60/100 judged
  70/100 judged
  80/100 judged
  90/100 judged
  100/100 judged
Done. Correctness: 19/100 correct → task6_correctness.json


In [5]:
# ── Faithfulness: Ragas Faithfulness on the first 20 sorted by financebench_id ──
# Slow: multiple LLM calls per sample. Cached per fb_id so re-runs only fill gaps.
import sys
!{sys.executable} -m pip install -q nltk 

try:
    from ragas.llms import llm_factory
    from ragas.metrics import Faithfulness
    from ragas.dataset_schema import SingleTurnSample
except ImportError:
    get_ipython().system(f'{sys.executable} -m pip install -q ragas')
    from ragas.llms import llm_factory
    from ragas.metrics import Faithfulness
    from ragas.dataset_schema import SingleTurnSample

# Ragas uses NLTK under the hood for sentence tokenization.
try:
    import nltk
    nltk.data.find("tokenizers/punkt_tab")
except Exception:
    import nltk
    nltk.download("punkt_tab", quiet=True)
    nltk.download("punkt", quiet=True)

T6_FAITH_CACHE = "task6_faithfulness.json"

t6_ragas_client = OpenAI(api_key=T6_NEBIUS_KEY, base_url=T6_NEBIUS_BASE)
t6_ragas_llm = llm_factory(model=T6_JUDGE_MODEL, client=t6_ragas_client)
t6_faith_metric = Faithfulness(llm=t6_ragas_llm)

t6_faith = json.loads(Path(T6_FAITH_CACHE).read_text()) if Path(T6_FAITH_CACHE).exists() else {}
t6_first20 = t6_df.head(20)
print(f"Starting with {len(t6_faith)}/{len(t6_first20)} cached.")

for _, row in t6_first20.iterrows():
    fb = row["financebench_id"]
    if fb in t6_faith:
        continue
    entry = t6_rag.get(fb)
    if entry is None:
        print(f"  [{fb}] no RAG answer yet — run the RAG cell first")
        continue
    contexts = [c["content"] for c in entry["retrieved_chunks"]] or ["[NO CONTEXT]"]
    sample = SingleTurnSample(
        user_input=row["question"],
        response=entry["answer"],
        retrieved_contexts=contexts,
    )
    try:
        score = t6_faith_metric.single_turn_score(sample)
        t6_faith[fb] = float(score) if score is not None else None
    except Exception as e:
        print(f"  [{fb}] faithfulness error: {type(e).__name__}: {e}")
        t6_faith[fb] = None
    Path(T6_FAITH_CACHE).write_text(json.dumps(t6_faith, indent=1))
    print(f"  [{fb}] faithfulness={t6_faith[fb]}")

vals = [v for v in t6_faith.values() if v is not None]
mean = sum(vals) / len(vals) if vals else float("nan")
print(f"Done. Faithfulness: mean={mean:.3f} over n={len(vals)} valid → {T6_FAITH_CACHE}")


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


/var/folders/lg/vvqm3z4111l_5k_q7hn8kcmc0000gn/T/ipykernel_53540/2959730533.py:8: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness


Starting with 0/20 cached.
  [financebench_id_00005] faithfulness=0.375
  [financebench_id_00070] faithfulness=0.7142857142857143
  [financebench_id_00080] faithfulness=1.0
  [financebench_id_00206] faithfulness=0.0
  [financebench_id_00215] faithfulness=1.0
  [financebench_id_00216] faithfulness=1.0
  [financebench_id_00222] faithfulness=0.0
  [financebench_id_00283] faithfulness=1.0
  [financebench_id_00288] faithfulness=0.6666666666666666
  [financebench_id_00299] faithfulness=0.0
  [financebench_id_00302] faithfulness=0.0
  [financebench_id_00382] faithfulness=1.0
  [financebench_id_00394] faithfulness=0.0
  [financebench_id_00407] faithfulness=0.6666666666666666
  [financebench_id_00438] faithfulness=0.0
  [financebench_id_00460] faithfulness=0.75
  [financebench_id_00464] faithfulness=1.0
  [financebench_id_00476] faithfulness=0.0
  [financebench_id_00494] faithfulness=1.0
  [financebench_id_00499] faithfulness=0.0
Done. Faithfulness: mean=0.509 over n=20 valid → task6_faithfulne

In [3]:
# ── Task 6 aggregates ── standalone: loads everything from cache files ──
import os, json
import pandas as pd
import numpy as np
from pathlib import Path
from datasets import load_dataset

T6_PAGE_HIT_K  = [1, 3, 5]
T6_RAG_CACHE   = "task6_rag.json"
T6_CORR_CACHE  = "task6_correctness.json"
T6_FAITH_CACHE = "task6_faithfulness.json"

def _parse_evidence(ev):
    if isinstance(ev, str):
        ev = json.loads(ev)
    if isinstance(ev, np.ndarray):
        ev = ev.tolist()
    if not isinstance(ev, list):
        ev = [ev]
    return [int(e.get("evidence_page_num", -1)) if isinstance(e, dict) else int(dict(e).get("evidence_page_num", -1)) for e in ev]

t6_df = load_dataset("PatronusAI/financebench", split="train").to_pandas()
t6_df = t6_df[t6_df["question_type"] != "metrics-generated"].copy()
t6_df = t6_df.rename(columns={"answer": "ground_truth"})
t6_df["evidence_page_nums"] = t6_df["evidence"].apply(_parse_evidence)
t6_df = t6_df.sort_values("financebench_id").reset_index(drop=True)

t6_rag   = json.loads(Path(T6_RAG_CACHE).read_text())   if Path(T6_RAG_CACHE).exists()   else {}
t6_corr  = json.loads(Path(T6_CORR_CACHE).read_text())  if Path(T6_CORR_CACHE).exists()  else {}
t6_faith = json.loads(Path(T6_FAITH_CACHE).read_text()) if Path(T6_FAITH_CACHE).exists() else {}

t6_rows = []
for _, row in t6_df.iterrows():
    fb = row["financebench_id"]
    if fb not in t6_rag or fb not in t6_corr:
        continue
    top5 = t6_rag[fb]["top5_pages"]
    ev_pages = set(row["evidence_page_nums"])
    ev_doc = row["doc_name"]
    hits = {}
    for k in T6_PAGE_HIT_K:
        hits[f"page_hit_at_{k}"] = int(any(
            p.get("doc_name") == ev_doc and p.get("page_number") in ev_pages
            for p in top5[:k]
        ))
    t6_rows.append({
        "financebench_id": fb,
        "question":        row["question"],
        "correctness":     1 if t6_corr[fb]["verdict"] == "correct" else 0,
        "faithfulness":    t6_faith.get(fb),
        **hits,
    })

task6_out = pd.DataFrame(t6_rows)

n = len(task6_out)
corr_n    = int(task6_out["correctness"].sum())
corr_mean = task6_out["correctness"].mean()
faith_vals = task6_out["faithfulness"].dropna()
faith_mean = faith_vals.mean() if len(faith_vals) else float("nan")

summary_rows = [
    {"metric": "correctness",  "n": n,               "value": round(corr_mean, 3),  "hits": corr_n, "out_of": n},
    {"metric": "faithfulness", "n": len(faith_vals),  "value": round(faith_mean, 3), "hits": "",     "out_of": ""},
]
for k in T6_PAGE_HIT_K:
    col = f"page_hit_at_{k}"
    h = int(task6_out[col].sum())
    summary_rows.append({"metric": f"page-hit@{k}", "n": n, "value": round(task6_out[col].mean(), 3), "hits": h, "out_of": n})

summary_df = pd.DataFrame(summary_rows)

# ── write: summary at top, blank row, then per-question data (single sheet) ──
T6_OUT = "assignment2_evaluation_ilay_nisim_boris_kogan.xlsx"
with pd.ExcelWriter(T6_OUT, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="evaluation", index=False, startrow=0)
    blank_row = len(summary_df) + 2          # +1 header +1 blank
    task6_out.to_excel(writer, sheet_name="evaluation", index=False, startrow=blank_row)
print(f"Saved → {T6_OUT}  (summary at top, {n} per-question rows below)")

print()
print("── average correctness, average faithfulness, and page-hit@k ──")
print(f"  correctness  (n={n}):   {corr_mean:.3f}  ({corr_n}/{n})")
print(f"  faithfulness (n={len(faith_vals)}):  {faith_mean:.3f}")
for k in T6_PAGE_HIT_K:
    col = f"page_hit_at_{k}"
    h = int(task6_out[col].sum())
    print(f"  page-hit@{k}  (n={n}):   {task6_out[col].mean():.3f}  ({h}/{n})")

task6_out.head(5)


/Users/YOUR_USER/dev/cloud/nebius/lesson5_rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saved → assignment2_evaluation_ilay_nisim_boris_kogan.xlsx  (summary at top, 100 per-question rows below)

── average correctness, average faithfulness, and page-hit@k ──
  correctness  (n=100):   0.190  (19/100)
  faithfulness (n=20):  0.509
  page-hit@1  (n=100):   0.200  (20/100)
  page-hit@3  (n=100):   0.330  (33/100)
  page-hit@5  (n=100):   0.400  (40/100)


,financebench_id,question,correctness,faithfulness,page_hit_at_1,page_hit_at_3,page_hit_at_5
0,financebench_id_00005,Does Corning have positive working capital bas...,0,0.375000,0,0,0
1,financebench_id_00070,Does American Water Works have positive workin...,0,0.714286,1,1,1
2,financebench_id_00080,Does Paypal have positive working capital base...,0,1.000000,0,0,0
3,financebench_id_00206,Are JPM's gross margins historically consisten...,0,0.000000,0,0,0
4,financebench_id_00215,Is Verizon a capital intensive business based ...,0,1.000000,0,1,1


---
## Task 7 — improvement cycles (15 points)

Two experiments that each vary **one thing at a time**.
1. **Experiment 1 — chunk_size=1500.** Rebuild the FAISS index with 1500-char chunks (overlap 225, same 15% ratio). Everything else identical to baseline.
2. **Experiment 2 — HyDE query reformulation.** Keep the baseline FAISS index. Before retrieval, ask the generator to write a short hypothetical answer and retrieve against `question + hypothetical` instead of the question alone. Motivated by the Task 5 small-vector sanity check, which showed the main failure mode on this dataset is vocabulary mismatch (e.g., "gross margin" doesn't literally appear in JPM's 10-K).
3. Before trying chunk_size 1500, i tried chunk_size 500, it is not included in code, but the data is included in `assignment2_improvement_cycles_*.xlsx` for comparison;(cached results only).

No reranker — both the assignment author's note (line 208) and the Task 5 finding indicate cross-encoder reranking would aggravate the vocabulary-mismatch problem.

Each experiment caches its RAG answers, judge verdicts, and faithfulness scores to disk per-question, . **The Task 7 cells below are self-contained** — you do not need to run Tasks 1–6 in this kernel first.

### Task 7 **Hypothesis** - which metric do you expect to improve
I assume that increasing chunk-size might improve a bit but not a lot — based on the small-vector experiment we saw that the issue is vocabulary mismatch: retrieval doesn't "translate" the terms in the query into the data it needs to fetch from the documents.
Therefore I expect chunk-size not to have a big effect. 
However I do expect HyDE to solve this issue by changing the query to have better retrieval. This should improve page-hit@k and therefore correctness and faithfulness. 


In [11]:
# ── Task 7 setup: standalone (does not depend on Tasks 1–6 kernel state) ─────
import os, json, sys
import pandas as pd
import numpy as np
from pathlib import Path
from openai import OpenAI
from datasets import load_dataset
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

load_dotenv()

T7_NEBIUS_KEY  = os.environ["NEBIUS_API_KEY"]
T7_NEBIUS_BASE = "https://api.studio.nebius.ai/v1/"
T7_GEN_MODEL   = "meta-llama/Llama-3.3-70B-Instruct"
T7_JUDGE_MODEL = "deepseek-ai/DeepSeek-V3.2"
T7_EMBED_MODEL = "BAAI/bge-small-en-v1.5"
T7_FAISS_DIR   = "faiss_chunk1000"
T7_RAG_K       = 4
T7_PAGE_HIT_K  = [1, 3, 5]

t7_nebius = OpenAI(api_key=T7_NEBIUS_KEY, base_url=T7_NEBIUS_BASE)

# ── dataset (same filter/parse/sort as Tasks 1–6) ────────────────────────────
def _t7_parse_evidence(ev):
    if isinstance(ev, str):
        ev = json.loads(ev)
    if isinstance(ev, np.ndarray):
        ev = ev.tolist()
    if not isinstance(ev, list):
        ev = [ev]
    pages, texts = [], []
    for e in ev:
        if not isinstance(e, dict):
            e = dict(e)
        pages.append(int(e.get("evidence_page_num", -1)))
        texts.append(str(e.get("evidence_text", "")))
    return pages, texts

t7_df = load_dataset("PatronusAI/financebench", split="train").to_pandas()
t7_df = t7_df[t7_df["question_type"] != "metrics-generated"].copy()
t7_df = t7_df.rename(columns={"answer": "ground_truth"})
_parsed = t7_df["evidence"].apply(_t7_parse_evidence)
t7_df["evidence_page_nums"] = _parsed.apply(lambda x: x[0])
t7_df["evidence_texts"]     = _parsed.apply(lambda x: x[1])
t7_df = t7_df.sort_values("financebench_id").reset_index(drop=True)

t7_doc_meta = (
    t7_df[["doc_name", "company", "doc_period"]]
    .drop_duplicates(subset="doc_name")
    .set_index("doc_name")
    .to_dict(orient="index")
)

# ── baseline FAISS (chunk_size=1000) ─────────────────────────────────────────
t7_embed = HuggingFaceEmbeddings(
    model_name=T7_EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
t7_vs = FAISS.load_local(T7_FAISS_DIR, t7_embed, allow_dangerous_deserialization=True)

# ── shared generation prompt (same as Task 6) ────────────────────────────────
T7_RAG_SYS = (
    "Follow these rules strictly:\n"
    "1. Answer ONLY from the provided context below. Do not use any outside knowledge.\n"
    "2. If the context does not contain the answer, say so explicitly — reply exactly: "
    "\"The provided context does not contain the answer.\" Do not guess.\n"
    "3. Keep answers concise — one short paragraph at most.\n"
    "4. Cite the source for each fact by appending the doc_name in parentheses.\n"
)

def _t7_format_ctx(chunks):
    if not chunks:
        return "[NO CONTEXT RETRIEVED]"
    return "\n\n---\n\n".join(
        f"[Chunk {i} | doc_name={c.metadata.get('doc_name')} | "
        f"page_number={c.metadata.get('page_number')}]\n{c.page_content.strip()}"
        for i, c in enumerate(chunks, 1)
    )

def _t7_generate(query, chunks):
    ctx = _t7_format_ctx(chunks)
    resp = t7_nebius.chat.completions.create(
        model=T7_GEN_MODEL,
        messages=[
            {"role": "system", "content": T7_RAG_SYS},
            {"role": "user",   "content": f"Context:\n{ctx}\n\nQuestion: {query}"},
        ],
        temperature=0.1,
        max_tokens=500,
    )
    return {
        "answer": resp.choices[0].message.content.strip(),
        "retrieved_chunks": [
            {"doc_name": c.metadata.get("doc_name"),
             "page_number": c.metadata.get("page_number"),
             "content": c.page_content}
            for c in chunks
        ],
    }

# ── judge (same prompt as Task 6) ────────────────────────────────────────────
T7_JUDGE_SYS = (
    "You are a strict financial-QA judge. Given a question, a ground-truth answer, "
    "and a candidate answer, decide whether the candidate is correct.\n"
    "Return STRICTLY as JSON: "
    "{\"verdict\": \"correct\" | \"incorrect\", \"justification\": \"<one short sentence>\"}.\n"
    "Rules:\n"
    "- A candidate is \"correct\" only if it conveys the same factual claim as the "
    "ground truth (minor wording differences ok).\n"
    "- Numbers must match (small rounding ok); directions (positive/negative, grew/shrunk) "
    "must match.\n"
    "- If the candidate refuses, says it has no answer, or is off-topic, mark it \"incorrect\".\n"
    "- Use only the ground truth — do not apply outside knowledge."
)

def _t7_judge(question, gt, candidate):
    resp = t7_nebius.chat.completions.create(
        model=T7_JUDGE_MODEL,
        messages=[
            {"role": "system", "content": T7_JUDGE_SYS},
            {"role": "user",
             "content": f"Question: {question}\n\nGround truth:\n{gt}\n\nCandidate answer:\n{candidate}"},
        ],
        temperature=0.0,
        max_tokens=200,
    )
    raw = resp.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if raw.lower().startswith("json"):
            raw = raw[4:].strip()
    try:
        parsed = json.loads(raw)
        verdict = parsed.get("verdict", "incorrect").lower()
        if verdict not in ("correct", "incorrect"):
            verdict = "incorrect"
        return {"verdict": verdict, "justification": parsed.get("justification", "")}
    except Exception:
        low = raw.lower()
        v = "correct" if ('"correct"' in low or low.startswith("correct")) and "incorrect" not in low else "incorrect"
        return {"verdict": v, "justification": raw[:200]}

# ── Ragas faithfulness (same DeepSeek judge wrapped for Ragas) ───────────────
try:
    from ragas.llms import llm_factory
    from ragas.metrics import Faithfulness
    from ragas.dataset_schema import SingleTurnSample
except ImportError:
    get_ipython().system(f"{sys.executable} -m pip install -q ragas")
    from ragas.llms import llm_factory
    from ragas.metrics import Faithfulness
    from ragas.dataset_schema import SingleTurnSample

try:
    import nltk
    nltk.data.find("tokenizers/punkt_tab")
except ModuleNotFoundError:
    get_ipython().system(f"{sys.executable} -m pip install -q nltk")
    import nltk
    nltk.download("punkt_tab", quiet=True)
    nltk.download("punkt", quiet=True)
except Exception:
    import nltk
    nltk.download("punkt_tab", quiet=True)
    nltk.download("punkt", quiet=True)

t7_ragas_client = OpenAI(api_key=T7_NEBIUS_KEY, base_url=T7_NEBIUS_BASE)
t7_ragas_llm    = llm_factory(model=T7_JUDGE_MODEL, client=t7_ragas_client)
t7_faith_metric = Faithfulness(llm=t7_ragas_llm)

# ── generic experiment runner: RAG → judge → faithfulness, all cached ───────
def run_experiment(name, rag_fn, k_for_pagehit=5):
    """rag_fn(question) -> {'answer': str, 'retrieved_chunks': [{doc_name, page_number, content}, ...]}"""
    rag_path   = f"task7_{name}_rag.json"
    corr_path  = f"task7_{name}_corr.json"
    faith_path = f"task7_{name}_faith.json"

    rag   = json.loads(Path(rag_path).read_text())   if Path(rag_path).exists()   else {}
    corr  = json.loads(Path(corr_path).read_text())  if Path(corr_path).exists()  else {}
    faith = json.loads(Path(faith_path).read_text()) if Path(faith_path).exists() else {}

    print(f"[{name}] start — rag={len(rag)}/{len(t7_df)}  corr={len(corr)}/{len(t7_df)}  faith={len(faith)}/20")

    # 1) RAG + retrieval snapshot (all 100)
    for _, row in t7_df.iterrows():
        fb = row["financebench_id"]
        if fb in rag:
            continue
        out = rag_fn(row["question"])
        rag[fb] = {
            "answer": out["answer"],
            "retrieved_chunks": out["retrieved_chunks"],
            "top5_pages": [
                {"doc_name": c["doc_name"], "page_number": c["page_number"]}
                for c in out["retrieved_chunks"][:k_for_pagehit]
            ],
        }
        Path(rag_path).write_text(json.dumps(rag, indent=1))
        print(f"  [{name}/rag] {fb}")

    # 2) Correctness (all 100)
    for _, row in t7_df.iterrows():
        fb = row["financebench_id"]
        if fb in corr:
            continue
        corr[fb] = _t7_judge(row["question"], row["ground_truth"], rag[fb]["answer"])
        Path(corr_path).write_text(json.dumps(corr, indent=1))
        print(f"  [{name}/corr] {fb} → {corr[fb]['verdict']}")

    # 3) Faithfulness (first 20 sorted by financebench_id — same set as Task 6)
    for _, row in t7_df.head(20).iterrows():
        fb = row["financebench_id"]
        if fb in faith:
            continue
        contexts = [c["content"] for c in rag[fb]["retrieved_chunks"]] or ["[NO CONTEXT]"]
        sample = SingleTurnSample(
            user_input=row["question"],
            response=rag[fb]["answer"],
            retrieved_contexts=contexts,
        )
        try:
            s = t7_faith_metric.single_turn_score(sample)
            faith[fb] = float(s) if s is not None else None
        except Exception as e:
            print(f"  [{name}/faith] {fb} error: {type(e).__name__}: {e}")
            faith[fb] = None
        Path(faith_path).write_text(json.dumps(faith, indent=1))
        print(f"  [{name}/faith] {fb} → {faith[fb]}")

    print(f"[{name}] done.")
    return rag, corr, faith

print(f"Task 7 setup OK. Dataset: {len(t7_df)} rows. "
      f"FAISS (chunk_size=1000): {t7_vs.index.ntotal} vectors.")
# ── aggregate printer: same format as Task 6 ────────────────────────────────
def print_aggregates(label, rag_path, corr_path, faith_path):
    rag   = json.loads(Path(rag_path).read_text())   if Path(rag_path).exists()   else {}
    corr  = json.loads(Path(corr_path).read_text())  if Path(corr_path).exists()  else {}
    faith = json.loads(Path(faith_path).read_text()) if Path(faith_path).exists() else {}

    hits = {k: 0 for k in T7_PAGE_HIT_K}
    corr_n, n = 0, 0
    for _, row in t7_df.iterrows():
        fb = row["financebench_id"]
        if fb not in rag or fb not in corr:
            continue
        n += 1
        if corr[fb]["verdict"] == "correct":
            corr_n += 1
        top5 = rag[fb]["top5_pages"]
        ev_pages = set(row["evidence_page_nums"])
        ev_doc = row["doc_name"]
        for k in T7_PAGE_HIT_K:
            if any(p.get("doc_name") == ev_doc and p.get("page_number") in ev_pages for p in top5[:k]):
                hits[k] += 1

    faith_vals = [v for v in faith.values() if v is not None]
    faith_mean = sum(faith_vals) / len(faith_vals) if faith_vals else float("nan")
    corr_mean  = corr_n / n if n else float("nan")

    print(f"── {label} ──")
    print(f"  correctness  (n={n}):   {corr_mean:.3f}  ({corr_n}/{n})")
    print(f"  faithfulness (n={len(faith_vals)}):  {faith_mean:.3f}")
    for k in T7_PAGE_HIT_K:
        rate = hits[k] / n if n else float("nan")
        print(f"  page-hit@{k}  (n={n}):   {rate:.3f}  ({hits[k]}/{n})")

print()
print_aggregates("Task 6 baseline (from cache)",
                 "task6_rag.json", "task6_correctness.json", "task6_faithfulness.json")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15218.92it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/var/folders/lg/vvqm3z4111l_5k_q7hn8kcmc0000gn/T/ipykernel_63782/613916274.py:149: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness


Task 7 setup OK. Dataset: 100 rows. FAISS (chunk_size=1000): 24218 vectors.

── Task 6 baseline (from cache) ──
  correctness  (n=100):   0.190  (19/100)
  faithfulness (n=20):  0.509
  page-hit@1  (n=100):   0.200  (20/100)
  page-hit@3  (n=100):   0.330  (33/100)
  page-hit@5  (n=100):   0.400  (40/100)


### Experiment 1 — chunk_size=1500 **Hypothesis.**  

**Hypothesis.** FinanceBench questions often hinge on single numbers inside tables (balance sheets, income statements). 1000-char chunks can split a table header from its rows, or a row label from its value, so the generator retrieves the number but can't interpret it. Enlarging chunks to 1500 (overlap 225, same 15% ratio) should keep more tables intact and increase the odds that question vocabulary overlaps *some* part of the chunk — the dominant failure mode on this dataset per Task 5. 1500 is the largest safe size: `BAAI/bge-small-en-v1.5` has a 512-token limit, so chunks much larger risk silent truncation during embedding.

We previously tried chunk_size=500 (not included in notebook) as exp1 and every metric got worse — correctness 0.190→0.150, faithfulness 0.509→0.340, page-hit@1/3/5 0.200/0.330/0.400 → 0.170/0.300/0.360 — consistent with fragmenting tables. This swap tests the opposite direction.

In [12]:
# Requires the Task 7 setup cell above to have been run in this kernel session
# (defines t7_df, t7_vs, t7_embed, t7_doc_meta, run_experiment, print_aggregates, etc.).
if "run_experiment" not in globals():
    raise RuntimeError(
        "Task 7 setup not initialized. Scroll up and run the '── Task 7 setup' cell "
        "first (it's fast — no LLM calls — just loads FAISS and defines helpers). "
        "After that, this cell runs standalone."
    )

# ── Experiment 1: rebuild FAISS at chunk_size=1500, rerun all metrics ────────
FAISS1500_DIR = "faiss_chunk1500"

if not Path(FAISS1500_DIR).exists():
    import urllib.request, urllib.error
    from langchain_community.document_loaders import PyPDFLoader
    try:
        from langchain_text_splitters import RecursiveCharacterTextSplitter
    except ImportError:
        get_ipython().system(f"{sys.executable} -m pip install -q langchain-text-splitters")
        from langchain_text_splitters import RecursiveCharacterTextSplitter

    PDF_DIR = Path("pdfs")
    PDF_DIR.mkdir(exist_ok=True)
    GITHUB_PDF_BASE = "https://raw.githubusercontent.com/patronus-ai/financebench/main/pdfs"

    needed = sorted(t7_df["doc_name"].unique())
    for name in needed:
        dest = PDF_DIR / f"{name}.pdf"
        if dest.exists() and dest.stat().st_size > 0:
            continue
        try:
            urllib.request.urlretrieve(f"{GITHUB_PDF_BASE}/{name}.pdf", dest)
        except urllib.error.HTTPError:
            if dest.exists():
                dest.unlink()

    docs_per_page = []
    for name in needed:
        path = PDF_DIR / f"{name}.pdf"
        if not path.exists():
            continue
        meta = t7_doc_meta[name]
        for p in PyPDFLoader(str(path)).load():
            p.metadata = {
                "doc_name":    name,
                "company":     meta["company"],
                "doc_period":  meta["doc_period"],
                "page_number": int(p.metadata.get("page", 0)),
            }
            docs_per_page.append(p)

    splitter1500 = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=225)
    chunks1500 = splitter1500.split_documents(docs_per_page)
    print(f"Split {len(docs_per_page)} pages → {len(chunks1500)} chunks (chunk_size=1500)")

    t7_vs1500 = FAISS.from_documents(chunks1500, t7_embed)
    t7_vs1500.save_local(FAISS1500_DIR)
    print(f"Saved → {FAISS1500_DIR}/  ({t7_vs1500.index.ntotal} vectors)")
else:
    t7_vs1500 = FAISS.load_local(FAISS1500_DIR, t7_embed, allow_dangerous_deserialization=True)
    print(f"Loaded {FAISS1500_DIR}/  ({t7_vs1500.index.ntotal} vectors)")

def rag_exp1_chunk1500(query, k=T7_RAG_K):
    chunks = t7_vs1500.similarity_search(query, k=max(k, 5))   # keep ≥5 for page-hit@5
    out = _t7_generate(query, chunks[:k])
    out["retrieved_chunks"] = [
        {"doc_name": c.metadata.get("doc_name"),
         "page_number": c.metadata.get("page_number"),
         "content": c.page_content}
        for c in chunks[:max(k, 5)]
    ]
    return out

exp1_rag, exp1_corr, exp1_faith = run_experiment("exp1_chunk1500", rag_exp1_chunk1500)

print()
print_aggregates("Experiment 1 (chunk_size=1500)",
                 "task7_exp1_chunk1500_rag.json",
                 "task7_exp1_chunk1500_corr.json",
                 "task7_exp1_chunk1500_faith.json")

Split 5448 pages → 16674 chunks (chunk_size=1500)
Saved → faiss_chunk1500/  (16674 vectors)
[exp1_chunk1500] start — rag=0/100  corr=0/100  faith=0/20
  [exp1_chunk1500/rag] financebench_id_00005
  [exp1_chunk1500/rag] financebench_id_00070
  [exp1_chunk1500/rag] financebench_id_00080
  [exp1_chunk1500/rag] financebench_id_00206
  [exp1_chunk1500/rag] financebench_id_00215
  [exp1_chunk1500/rag] financebench_id_00216
  [exp1_chunk1500/rag] financebench_id_00222
  [exp1_chunk1500/rag] financebench_id_00283
  [exp1_chunk1500/rag] financebench_id_00288
  [exp1_chunk1500/rag] financebench_id_00299
  [exp1_chunk1500/rag] financebench_id_00302
  [exp1_chunk1500/rag] financebench_id_00382
  [exp1_chunk1500/rag] financebench_id_00394
  [exp1_chunk1500/rag] financebench_id_00407
  [exp1_chunk1500/rag] financebench_id_00438
  [exp1_chunk1500/rag] financebench_id_00460
  [exp1_chunk1500/rag] financebench_id_00464
  [exp1_chunk1500/rag] financebench_id_00476
  [exp1_chunk1500/rag] financebench_id_

**Interpretation — Experiment 1 (chunk_size=1500):**

- Page-hit@k vs baseline (chunk=1000): got worse on every k (h@1 0.200→0.140, h@3 0.330→0.260, h@5 0.400→0.350).
- Correctness vs baseline: improved slightly (0.190 → 0.210, +2 questions).
- Faithfulness vs baseline: got worse (0.509 → 0.309).
- Context: the earlier chunk_size=500 attempt was also worse on every metric — fragmenting tables hurt. Going *larger* didn't help retrieval either: bigger chunks average more topics into a single embedding and dilute discrimination, so page-hit dropped. The small correctness bump comes from the subset where retrieval still hit the right page and the longer chunk gave the generator more table context. Faithfulness fell because longer chunks invite more unsupported claims per answer. Net read: chunk size is not the lever — vocabulary mismatch is.

### Experiment 2 — HyDE query reformulation

**Hypothesis.** Task 5's small-vector sanity check showed the dominant failure mode on this dataset is **vocabulary mismatch**: user questions use high-level financial terms ("gross margin", "working capital") that the filings don't use verbatim. HyDE addresses this directly — ask the generator for a short hypothetical answer, then retrieve against `question + hypothetical`. The hypothetical uses 10-K-style vocabulary (line-item names, dollar amounts), giving FAISS a better match against real pages. Expected: **page-hit@k rises**, correctness follows on the questions that were previously blocked by vocabulary mismatch. Faithfulness may improve simply because the generator now has relevant chunks to ground in.

In [14]:
# Requires the Task 7 setup cell above to have been run in this kernel session
# (defines t7_df, t7_vs, t7_embed, t7_doc_meta, run_experiment, print_aggregates, etc.).
if "run_experiment" not in globals():
    raise RuntimeError(
        "Task 7 setup not initialized. Scroll up and run the '── Task 7 setup' cell "
        "first (it's fast — no LLM calls — just loads FAISS and defines helpers). "
        "After that, this cell runs standalone."
    )

# ── Experiment 2: HyDE. Same FAISS, same prompt, same k; only retrieval query changes.
HYDE_SYS = (
    "You will receive a financial question that will be used to retrieve context from "
    "SEC filings (10-Ks, 10-Qs, 8-Ks, earnings releases). Write a SHORT (1–2 sentences) "
    "plausible hypothetical answer in the style of such a filing — name specific line "
    "items (e.g., 'current assets', 'gross profit', 'net revenue') and include plausible "
    "dollar figures even if you must guess. Do not refuse. Output only the hypothetical."
)

HYDE_CACHE_PATH = "task7_exp2_hyde_hypotheticals.json"
_hyde_cache = json.loads(Path(HYDE_CACHE_PATH).read_text()) if Path(HYDE_CACHE_PATH).exists() else {}

def hyde_rewrite(question):
    if question in _hyde_cache:
        return _hyde_cache[question]
    resp = t7_nebius.chat.completions.create(
        model=T7_GEN_MODEL,
        messages=[{"role":"system","content":HYDE_SYS},
                  {"role":"user","content":question}],
        temperature=0.3,
        max_tokens=140,
    )
    hyp = resp.choices[0].message.content.strip()
    _hyde_cache[question] = hyp
    Path(HYDE_CACHE_PATH).write_text(json.dumps(_hyde_cache, indent=1))
    return hyp

def rag_exp2_hyde(query, k=T7_RAG_K):
    hyp = hyde_rewrite(query)
    retrieval_text = f"{query}\n{hyp}"
    chunks = t7_vs.similarity_search(retrieval_text, k=max(k, 5))   # baseline FAISS (chunk_size=1000)
    out = _t7_generate(query, chunks[:k])     # generator still sees only the ORIGINAL question
    out["retrieved_chunks"] = [
        {"doc_name": c.metadata.get("doc_name"),
         "page_number": c.metadata.get("page_number"),
         "content": c.page_content}
        for c in chunks[:max(k, 5)]
    ]
    return out

exp2_rag, exp2_corr, exp2_faith = run_experiment("exp2_hyde", rag_exp2_hyde)

print()
print_aggregates("Experiment 2 (HyDE)",
                 "task7_exp2_hyde_rag.json",
                 "task7_exp2_hyde_corr.json",
                 "task7_exp2_hyde_faith.json")

[exp2_hyde] start — rag=0/100  corr=0/100  faith=0/20
  [exp2_hyde/rag] financebench_id_00005
  [exp2_hyde/rag] financebench_id_00070
  [exp2_hyde/rag] financebench_id_00080
  [exp2_hyde/rag] financebench_id_00206
  [exp2_hyde/rag] financebench_id_00215
  [exp2_hyde/rag] financebench_id_00216
  [exp2_hyde/rag] financebench_id_00222
  [exp2_hyde/rag] financebench_id_00283
  [exp2_hyde/rag] financebench_id_00288
  [exp2_hyde/rag] financebench_id_00299
  [exp2_hyde/rag] financebench_id_00302
  [exp2_hyde/rag] financebench_id_00382
  [exp2_hyde/rag] financebench_id_00394
  [exp2_hyde/rag] financebench_id_00407
  [exp2_hyde/rag] financebench_id_00438
  [exp2_hyde/rag] financebench_id_00460
  [exp2_hyde/rag] financebench_id_00464
  [exp2_hyde/rag] financebench_id_00476
  [exp2_hyde/rag] financebench_id_00494
  [exp2_hyde/rag] financebench_id_00499
  [exp2_hyde/rag] financebench_id_00517
  [exp2_hyde/rag] financebench_id_00521
  [exp2_hyde/rag] financebench_id_00540
  [exp2_hyde/rag] financeb

**Interpretation — Experiment 2 (HyDE):**

- Page-hit@k vs baseline: improved on every k (h@1 0.200→0.240, h@3 0.330→0.410, h@5 0.400→0.500).
- Correctness vs baseline: improved (0.190 → 0.230, +4 questions).
- Faithfulness vs baseline: got worse (0.509 → 0.437).
- Spot-check `00206` (JPM gross margin) — did the HyDE hypothetical surface the balance-sheet page? 
HyDE fixes the doc-level retrieval — top-5 went from zero JPMORGAN_2022_10K hits (baseline) to all 5 results coming from JPMorgan filings (with the 2022 10-K at rank #1).
But the specific evidence page (page 2) still wasn't retrieved — HyDE returned pages 48, 4, 15, 4, 166.
So: HyDE narrowed to the right corpus but not the right page; the question stayed incorrect. This matches Task 5's conclusion that "gross margin" is structurally hard for JPM because the term doesn't appear verbatim — even a HyDE-style reformulation didn't surface the income-statement page.

In [15]:
# ── Task 7 aggregation: baseline + 2 experiments → xlsx ──────────────────────
# Standalone: loads everything from JSON caches on disk.
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datasets import load_dataset

T7_PAGE_HIT_K = [1, 3, 5]

def _parse_ev(ev):
    if isinstance(ev, str):
        ev = json.loads(ev)
    if isinstance(ev, np.ndarray):
        ev = ev.tolist()
    if not isinstance(ev, list):
        ev = [ev]
    return [int(e.get("evidence_page_num", -1)) if isinstance(e, dict)
            else int(dict(e).get("evidence_page_num", -1)) for e in ev]

_df = load_dataset("PatronusAI/financebench", split="train").to_pandas()
_df = _df[_df["question_type"] != "metrics-generated"].copy()
_df = _df.rename(columns={"answer": "ground_truth"})
_df["evidence_page_nums"] = _df["evidence"].apply(_parse_ev)
_df = _df.sort_values("financebench_id").reset_index(drop=True)

def aggregate(name, rag_path, corr_path, faith_path, change_desc):
    rag   = json.loads(Path(rag_path).read_text())   if Path(rag_path).exists()   else {}
    corr  = json.loads(Path(corr_path).read_text())  if Path(corr_path).exists()  else {}
    faith = json.loads(Path(faith_path).read_text()) if Path(faith_path).exists() else {}

    per_q = []
    for _, row in _df.iterrows():
        fb = row["financebench_id"]
        if fb not in rag or fb not in corr:
            continue
        top5 = rag[fb]["top5_pages"]
        ev_pages = set(row["evidence_page_nums"])
        ev_doc = row["doc_name"]
        hits = {}
        for k in T7_PAGE_HIT_K:
            hits[f"page_hit_at_{k}"] = int(any(
                p.get("doc_name") == ev_doc and p.get("page_number") in ev_pages
                for p in top5[:k]
            ))
        per_q.append({
            "correctness":  1 if corr[fb]["verdict"] == "correct" else 0,
            "faithfulness": faith.get(fb),
            **hits,
        })

    df = pd.DataFrame(per_q)
    n = len(df)
    fv = df["faithfulness"].dropna()
    row = {
        "experiment":    name,
        "change":        change_desc,
        "correctness":   round(df["correctness"].mean(), 3) if n else float("nan"),
        "faithfulness":  round(fv.mean(), 3) if len(fv) else float("nan"),
    }
    for k in T7_PAGE_HIT_K:
        row[f"page_hit_at_{k}"] = round(df[f"page_hit_at_{k}"].mean(), 3) if n else float("nan")
    return row

rows = [
    aggregate("baseline (Task 6)", "task6_rag.json", "task6_correctness.json", "task6_faithfulness.json",
              "chunk=1000, k=4, question-only retrieval"),
    aggregate("exp1_chunk1500", "task7_exp1_chunk1500_rag.json", "task7_exp1_chunk1500_corr.json",
              "task7_exp1_chunk1500_faith.json", "chunk_size 1000 → 1500 (overlap 225)"),
    aggregate("exp2_hyde", "task7_exp2_hyde_rag.json", "task7_exp2_hyde_corr.json",
              "task7_exp2_hyde_faith.json", "HyDE: retrieve on question + hypothetical"),
]

task7_out = pd.DataFrame(rows)
T7_OUT = "assignment2_improvement_cycles_ilay_nisim_boris_kogan.xlsx"
task7_out.to_excel(T7_OUT, index=False)
print(f"Saved → {T7_OUT}")
print()
print(task7_out.to_string(index=False))

Saved → assignment2_improvement_cycles_ilay_nisim_boris_kogan.xlsx

       experiment                                    change  correctness  faithfulness  page_hit_at_1  page_hit_at_3  page_hit_at_5
baseline (Task 6)  chunk=1000, k=4, question-only retrieval         0.19         0.509           0.20           0.33           0.40
   exp1_chunk1500      chunk_size 1000 → 1500 (overlap 225)         0.21         0.309           0.14           0.26           0.35
        exp2_hyde HyDE: retrieve on question + hypothetical         0.23         0.437           0.24           0.41           0.50


### Wrap-up

**Where does the pipeline fail most — retrieval, generation, or both?**
The pipeline fails at page-hit, which propagates into the rest of the pipeline; this results in poor correctness and only medium faithfulness. So retrieval is the biggest failure, but it affects generation — i.e., both.
We see in the HyDE experiment that even improving retrieval didn't improve faithfulness, which means there is independent headroom to improve generation (the generator LLM and/or the prompt).

**If you had one more week, what would you try next?**
Candidate directions:
(a) combine HyDE with increasing chunk-size — they attack different bottlenecks (retrieval vocab vs. generator context per chunk), so gains may compound on correctness even if page-hit slips.
(b) improve the generation model + prompt, including a structured-output prompt that forces the model to cite the exact line item it used — since HyDE didn't improve faithfulness, generation is the next bottleneck. (merges the original "better generation" and "structured-output prompt" directions into one.)
(c) hybrid retrieval (BM25 + dense) — directly addresses vocabulary mismatch by matching surface forms.
(d) question decomposition for multi-hop numeric questions.
(e) a table-aware PDF loader — PyPDFLoader loses table structure entirely, and most FinanceBench evidence lives inside tables.

**Main takeaway — you can't build a good RAG without knowing your data.**
Every failure mode we hit traces back to a domain gap, not an algorithmic one:
- *Financial language*: questions ask about "gross margin" or "working capital" — terms that don't appear verbatim in the filings. Knowing this up front would have pushed us toward query reformulation (HyDE) or a finance-specific re-tokenization from the start.
- *PDF structure*: PyPDFLoader flattens tables, but the evidence for most FinanceBench questions lives *inside* tables. A table-aware loader is not an optimization — it's a prerequisite.
- *Specific fields*: to answer "working capital", the system needs to know it's `current assets − current liabilities`, retrieve both line items, and compute. A general-purpose retriever can't do this without either a finance-specific decomposer or pre-built field extraction over the filings.

Generic RAG tuning (chunk size, k, embeddings) gave us small, mostly-cancelling deltas. The lever that matters is encoding domain knowledge into the pipeline itself.